# 01. Build Dataset (raw → FuxiCTR parquet)

다양한 CTR 데이터셋을 동일한 노트북에서 `DATASET` 스위치만 바꿔 FuxiCTR parquet으로 변환한다.

흐름:
1. 데이터셋 선택 (`DATASET_REGISTRY`)
2. 원래 컴럼/샘플 들여다보고 사용할 피쳘 직접 지정
3. valid 파일 없으면 train에서 10% 분리 (group기반, 예: user_id)
4. dataset YAML 생성 → FuxiCTR `FeatureProcessor` + `build_dataset` 실행
5. 산출물 확인 (`data/<dataset_id>/feature_map.json`, `train.parquet`, …)

참고: `demo/example1_build_dataset_to_parquet.py`, `demo/example5_DeepFM_with_pretrained_emb_as_weights.py`.

In [8]:
import sys, os
from pathlib import Path
import pandas as pd

# utils 패키지를 import 하기 위해 notebook 디렉토리를 sys.path에 추가
_HERE = Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))
from utils import nb_utils as U
U.add_fuxictr_to_path()
print('PROJECT_ROOT =', U.PROJECT_ROOT)
print('DATA_ROOT    =', U.DATA_ROOT)

PROJECT_ROOT = /NAS/hyunwoong/Hayanmind-project
DATA_ROOT    = /NAS/hyunwoong/Hayanmind-project/data


## 1. DATASET_REGISTRY

데이터셋별 raw 파일 경로, 포맷, 사전 임베딩 파일, valid 생성용 그룹 키를 등록.
새 데이터셋은 아래 덕셔너리에 항목만 추가하면 이 노트북을 그대로 재사용할 수 있다.

In [ ]:
DATASET_REGISTRY = {
    'amazon_electronics': {
        'dataset_id': 'AmazonElectronics_x1',
        'raw_dir':    U.RAW_DATA_ROOT / 'AmazonElectronics_x1',
        'train_file': 'train.csv',
        'test_file':  'test.csv',
        'valid_file': None,            # None → train에서 10% 분리
        'data_format': 'csv',
        'label_name': 'label',
        'group_col':  'user_id',
        'pretrained_emb': {},
    },
    'kuaivideo': {
        'dataset_id': 'KuaiVideo_x1',
        'raw_dir':    U.RAW_DATA_ROOT / 'KuaiVideo_x1',
        'train_file': 'train.csv',
        'test_file':  'test.csv',
        'valid_file': None,
        'data_format': 'csv',
        'label_name': 'is_click',
        'group_col':  'user_id',
        'pretrained_emb': {
            'item_id': {'path': U.RAW_DATA_ROOT / 'KuaiVideo_x1' / 'item_visual_emb_dim64.h5', 'dim': 64},
            'user_id': {'path': U.RAW_DATA_ROOT / 'KuaiVideo_x1' / 'user_visual_emb_dim64.h5', 'dim': 64},
        },
    },
    'microvideo': {
        'dataset_id': 'MicroVideo1.7M_x1',
        'raw_dir':    U.RAW_DATA_ROOT / 'MicroVideo1.7M_x1',
        'train_file': 'train.csv',
        'test_file':  'test.csv',
        'valid_file': None,
        'data_format': 'csv',
        'label_name': 'is_click',
        'group_col':  'user_id',
        'pretrained_emb': {
            'item_id': {'path': U.RAW_DATA_ROOT / 'MicroVideo1.7M_x1' / 'item_image_emb_dim64.h5', 'dim': 64},
        },
    },
    'taobao_ads': {
        'dataset_id': 'TaobaoAd_x1',
        'raw_dir':    U.RAW_DATA_ROOT / 'TaobaoAd_x1',
        'train_file': 'train.csv',
        'test_file':  'test.csv',
        'valid_file': None,
        'data_format': 'csv',
        'label_name': 'clk',
        'group_col':  'userid',
        'pretrained_emb': {},
    },
    'taac2026': {
        'dataset_id': 'TAAC2026',
        'raw_dir':    U.RAW_DATA_ROOT / 'TAAC2026',
        'train_file': 'train.csv',
        'test_file':  'test.csv',
        'valid_file': None,
        'data_format': 'csv',
        'label_name': 'label',
        'group_col':  'user_id',
        'pretrained_emb': {
            # h5 파일명의 k값은 01_embedding.ipynb 실행 결과에 맞춰 수정
            'fid_61': {'path': U.RAW_DATA_ROOT / 'TAAC2026' / 'fid_61_k4.h5', 'dim': 256},
            'fid_87': {'path': U.RAW_DATA_ROOT / 'TAAC2026' / 'fid_87_k4.h5', 'dim': 320},
        },
    },
}
list(DATASET_REGISTRY)

## 2. 데이터셋 선택

다른 데이터셋을 돌리려면 이 셀의 `DATASET`만 바꾸고 상위부터 실행.

In [10]:
DATASET = 'amazon_electronics'
CFG = DATASET_REGISTRY[DATASET]
DATASET_ID = CFG['dataset_id']
RAW_DIR    = CFG['raw_dir']
TRAIN_RAW  = RAW_DIR / CFG['train_file']
TEST_RAW   = RAW_DIR / CFG['test_file']
VALID_RAW  = RAW_DIR / CFG['valid_file'] if CFG['valid_file'] else None
print(f'dataset_id   = {DATASET_ID}')
print(f'train_raw    = {TRAIN_RAW}  exists={TRAIN_RAW.exists()}')
print(f'test_raw     = {TEST_RAW}   exists={TEST_RAW.exists()}')
print(f'valid_raw    = {VALID_RAW}')
print(f'data_format  = {CFG["data_format"]}')

dataset_id   = AmazonElectronics_x1
train_raw    = /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1/train.csv  exists=True
test_raw     = /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1/test.csv   exists=True
valid_raw    = None
data_format  = csv


## 3. 원래 컴럼 / 샘플 미리보기

아래 표를 보고 다음 셀의 `FEATURE_COLS`를 직접 편집한다.
- `type`: `categorical`, `numeric`, `sequence` 중 선택
- 시퀀스 피쳘의 구분자는 `^` (FuxiCTR 기본)

In [11]:
print('--- raw columns ---')
print(U.load_raw_columns(TRAIN_RAW))
print('\n--- column overview (sample 10k) ---')
U.column_overview(TRAIN_RAW, sample_rows=10_000)

--- raw columns ---
['label', 'user_id', 'item_id', 'cate_id', 'item_history', 'cate_history']

--- column overview (sample 10k) ---


,column,dtype,n_unique(sample),null_rate(sample),samples
0,label,int64,2,0.0,"1, 1, 1"
1,user_id,int64,9472,0.0,"104760, 129282, 130232"
2,item_id,int64,8537,0.0,"18486, 4206, 21354"
3,cate_id,int64,574,0.0,"674, 463, 556"
4,item_history,object,9828,0.0,"3737^19450, 3647^4342^6855^3805, 1805^4309"
5,cate_history,object,8601,0.0,"288^196, 281^463^558^674, 87^87"


## 4. 피쳘 설정 셀 (수동 편집)

사용할 컴럼만 이 리스트에 남긴다. `active: False`로 두면 FuxiCTR가 무시한다.
사전 임베딩은 `PRETRAINED_EMB`에서 컴럼별로 머지된다 (demo/example5 패턴).

아래 템플릿은 `DATASET` 값에 따라 미리 제공. 필요 시 직접 수정.

In [ ]:
# 예시: AmazonElectronics_x1 (label,user_id,item_id,cate_id,item_history,cate_history)
FEATURE_TEMPLATES = {
    'amazon_electronics': {
        'feature_cols': [
            {'name': 'user_id',      'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'item_id',      'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'cate_id',      'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'item_history', 'active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id'},
            {'name': 'cate_history', 'active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'cate_id'},
        ],
        'label_col': {'name': 'label', 'dtype': 'float'},
    },
    'kuaivideo': {
        'feature_cols': [
            {'name': 'user_id',   'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'item_id',   'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'pos_items', 'active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id', 'padding': 'pre', 'na_value': ''},
            {'name': 'neg_items', 'active': False, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id', 'padding': 'pre', 'na_value': ''},
        ],
        'label_col': {'name': 'is_click', 'dtype': 'float'},
    },
    'microvideo': {
        'feature_cols': [
            {'name': 'user_id',           'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'item_id',           'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'cate_id',           'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'clicked_items',     'active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id', 'padding': 'pre', 'na_value': ''},
            {'name': 'clicked_categories','active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'cate_id', 'padding': 'pre', 'na_value': ''},
        ],
        'label_col': {'name': 'is_click', 'dtype': 'float'},
    },
    'taobao_ads': {
        # FuxiCTR BARS 표준 TaobaoAd_x1 스키마 (user/ad/context + click_sequence)
        'feature_cols': [
            {'name': 'userid',                 'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'adgroup_id',             'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'pid',                    'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'cate_id',                'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'campaign_id',            'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'customer',               'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'brand',                  'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'cms_segid',              'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'cms_group_id',           'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'final_gender_code',      'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'age_level',              'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'pvalue_level',           'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'shopping_level',         'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'occupation',             'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'new_user_class_level',   'active': True, 'dtype': 'str', 'type': 'categorical'},
            {'name': 'click_sequence',         'active': True, 'dtype': 'str', 'type': 'sequence',
             'splitter': '^', 'max_len': 50, 'share_embedding': 'adgroup_id', 'padding': 'pre', 'na_value': ''},
        ],
        'label_col': {'name': 'clk', 'dtype': 'float'},
    },
    'taac2026': {
        'feature_cols': (
            # --- ID features ---
            [{'name': 'user_id', 'active': True, 'dtype': 'str', 'type': 'categorical'},
             {'name': 'item_id', 'active': True, 'dtype': 'str', 'type': 'categorical'},
             {'name': 'timestamp', 'active': True, 'dtype': 'float', 'type': 'numeric'}]
            # --- User Int scalar (35) → categorical ---
            + [{'name': f'user_int_feats_{f}', 'active': True, 'dtype': 'str', 'type': 'categorical'}
               for f in [1, 3, 4] + list(range(48, 60)) + [82, 86] + list(range(92, 110))]
            # --- Item Int scalar (13) → categorical ---
            + [{'name': f'item_int_feats_{f}', 'active': True, 'dtype': 'str', 'type': 'categorical'}
               for f in [5, 6, 7, 8, 9, 10, 12, 13, 16, 81, 83, 84, 85]]
            # --- Unpaired User Int list (3) → sequence ---
            + [{'name': f'user_int_feats_{f}', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 15, 'na_value': ''}
               for f in [15, 60, 80]]
            # --- Item Int list (1) → sequence ---
            + [{'name': 'item_int_feats_11', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 20, 'na_value': ''}]
            # --- Domain A sequences (9) ---
            + [{'name': f'domain_a_seq_{i}', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 50, 'na_value': ''}
               for i in range(38, 47)]
            # --- Domain B sequences (14) ---
            + [{'name': f'domain_b_seq_{i}', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 50, 'na_value': ''}
               for i in list(range(67, 80)) + [88]]
            # --- Domain C sequences (12) ---
            + [{'name': f'domain_c_seq_{i}', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 50, 'na_value': ''}
               for i in list(range(27, 38)) + [47]]
            # --- Domain D sequences (10) ---
            + [{'name': f'domain_d_seq_{i}', 'active': True, 'dtype': 'str', 'type': 'sequence',
                'splitter': '^', 'max_len': 50, 'na_value': ''}
               for i in range(17, 27)]
            # --- Pseudo-category (from 01_embedding.ipynb) → categorical + pretrained_emb ---
            + [{'name': 'fid_61', 'active': True, 'dtype': 'str', 'type': 'categorical'},
               {'name': 'fid_87', 'active': True, 'dtype': 'str', 'type': 'categorical'}]
        ),
        'label_col': {'name': 'label', 'dtype': 'float'},
    },
}
tmpl = FEATURE_TEMPLATES[DATASET]
FEATURE_COLS = tmpl['feature_cols']
LABEL_COL    = tmpl['label_col']
PRETRAINED_EMB = CFG.get('pretrained_emb') or {}

# 사전 임베딩이 있으면 해당 칼럼에 병합 (demo/example5 패턴)
for fc in FEATURE_COLS:
    name = fc['name']
    if name in PRETRAINED_EMB:
        info = PRETRAINED_EMB[name]
        fc['pretrained_emb'] = U._rel_to_data_root(info['path'])
        fc['embedding_dim']  = info['dim']
        fc['freeze_emb']     = info.get('freeze', True)

print(f'FEATURE_COLS = {len(FEATURE_COLS)} cols')
for fc in FEATURE_COLS:
    print('  ', fc)
print('LABEL_COL =', LABEL_COL)

## 5. Valid 분리 (valid 파일이 없는 경우)

train에서 10%를 분리. `group_col`이 있으면 그룹 단위로 분리하여 사용자 누수 방지.

In [13]:
if VALID_RAW is None:
    split_dir = U.RAW_DATA_ROOT / f'{DATASET_ID}_split'
    TRAIN_RAW, VALID_RAW = U.split_train_valid(
        train_path=RAW_DIR / CFG['train_file'],
        out_dir=split_dir,
        valid_ratio=0.1,
        group_col=CFG.get('group_col'),
    )
print('TRAIN =', TRAIN_RAW)
print('VALID =', VALID_RAW)
print('TEST  =', TEST_RAW)

[split] reuse existing split at /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1_split
TRAIN = /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1_split/train.csv
VALID = /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1_split/valid.csv
TEST  = /NAS/hyunwoong/Hayanmind-project/data/raw_data/AmazonElectronics_x1/test.csv


## 6. dataset YAML 생성 & FuxiCTR build

산출물:
- `notebook/configs/datasets/<dataset_id>.yaml` (개별), `dataset_config.yaml` (통합)
- `data/<dataset_id>/feature_map.json`, `feature_processor.pkl`, `train.parquet`, `valid.parquet`, `test.parquet`

⚠️  FuxiCTR 주의점: `feature_map.json`이 이미 존재하면 `build_dataset`은 **parquet 변환까지 통째로 스킵**한다
(경고만 `logging.warn`으로 남김 → 쉽게 놀치먌). 이전 실행에서 feature 파일만 남고 parquet이 없으면
`FORCE_REBUILD = True`로 설정해 기존 artifact를 지우고 다시 빌드한다.
`build_from_yaml`은 빌드 종료 후 parquet 파일이 실제로 생겼는지 검증하고 없으면 명확하게 에러를 낸다.

In [14]:
assert FEATURE_COLS, f'FEATURE_COLS 비어있음. FEATURE_TEMPLATES[{DATASET!r}] 편집 필요.'

# 기존 feature_map.json이 있고 parquet이 없는 상황이면 True로 바꿔 전체 재빌드
FORCE_REBUILD = False

U.materialize_dataset_yaml(
    dataset_id=DATASET_ID,
    feature_cols=FEATURE_COLS,
    label_col=LABEL_COL,
    train_data=TRAIN_RAW,
    valid_data=VALID_RAW,
    test_data=TEST_RAW,
    data_format=CFG['data_format'],
    min_categr_count=1,
)

params = U.build_from_yaml(DATASET_ID, force_rebuild=FORCE_REBUILD)
print('[done] dataset_id =', DATASET_ID)

2026-04-15 16:29:19,991 P1385279 INFO FuxiCTR version: 2.3.9
2026-04-15 16:29:19,992 P1385279 INFO Build params: {
    "data_format": "csv",
    "data_root": "../data/",
    "dataset_id": "AmazonElectronics_x1",
    "feature_cols": "[{'name': 'user_id', 'active': True, 'dtype': 'str', 'type': 'categorical'}, {'name': 'item_id', 'active': True, 'dtype': 'str', 'type': 'categorical'}, {'name': 'cate_id', 'active': True, 'dtype': 'str', 'type': 'categorical'}, {'name': 'item_history', 'active': True, 'dtype': 'str', 'type': 'sequence', 'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id'}, {'name': 'cate_history', 'active': True, 'dtype': 'str', 'type': 'sequence', 'splitter': '^', 'max_len': 50, 'share_embedding': 'cate_id'}]",
    "label_col": "{'name': 'label', 'dtype': 'float'}",
    "min_categr_count": "1",
    "model_id": "AmazonElectronics_x1_build",
    "model_root": "/NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs",
    "test_data": "../data/raw_data/AmazonElectr

[yaml] wrote /NAS/hyunwoong/Hayanmind-project/notebook/configs/datasets/AmazonElectronics_x1.yaml


100%|██████████| 3/3 [00:01<00:00,  2.25it/s]
2026-04-15 16:29:22,129 P1385279 INFO Processing column: {'name': 'item_id', 'active': True, 'dtype': 'str', 'type': 'categorical'}
100%|██████████| 3/3 [00:00<00:00,  4.41it/s]
2026-04-15 16:29:23,629 P1385279 INFO Processing column: {'name': 'cate_id', 'active': True, 'dtype': 'str', 'type': 'categorical'}
100%|██████████| 3/3 [00:00<00:00, 17.16it/s]
2026-04-15 16:29:24,233 P1385279 INFO Processing column: {'name': 'item_history', 'active': True, 'dtype': 'str', 'type': 'sequence', 'splitter': '^', 'max_len': 50, 'share_embedding': 'item_id'}
100%|██████████| 3/3 [00:05<00:00,  1.90s/it]
2026-04-15 16:29:31,556 P1385279 INFO Processing column: {'name': 'cate_history', 'active': True, 'dtype': 'str', 'type': 'sequence', 'splitter': '^', 'max_len': 50, 'share_embedding': 'cate_id'}
100%|██████████| 3/3 [00:04<00:00,  1.48s/it]
2026-04-15 16:29:37,365 P1385279 INFO Set column index...
2026-04-15 16:29:37,367 P1385279 INFO Save feature_map t

[build] feature files: 3  parquet files: 3
  - AmazonElectronics_x1/test.parquet
  - AmazonElectronics_x1/train.parquet
  - AmazonElectronics_x1/valid.parquet
[build] done -> /NAS/hyunwoong/Hayanmind-project/data/AmazonElectronics_x1
[done] dataset_id = AmazonElectronics_x1


## 7. 산출물 검증

In [15]:
fmap = U.load_feature_map(DATASET_ID)
print('num_fields   =', fmap['num_fields'])
print('total_features =', fmap['total_features'])
print('labels       =', fmap['labels'])
print('features (첫 5개):')
for f in fmap['features'][:5]:
    print(' ', f)

ds_dir = U.DATA_ROOT / DATASET_ID
parquet_paths = sorted(Path(ds_dir).rglob('*.parquet'))
print('\nparquet files:')
for p in parquet_paths:
    print('  ', p.relative_to(U.PROJECT_ROOT))

if parquet_paths:
    sample = pd.read_parquet(parquet_paths[0])
    print('\nsample head:')
    display(sample.head())
    print('shape:', sample.shape)
    seq_cols = [f['name'] if isinstance(f, dict) and 'name' in f else list(f.keys())[0]
                for f in fmap['features']
                if (list(f.values())[0].get('type') == 'sequence')]
    for c in seq_cols:
        if c in sample.columns:
            lens = sample[c].apply(lambda x: 0 if x is None else len(x))
            print(f'{c}: len p50={lens.quantile(0.5):.0f}, p90={lens.quantile(0.9):.0f}, max={lens.max()}')

num_fields   = 5
total_features = 236971
labels       = ['label']
features (첫 5개):
  {'user_id': {'source': '', 'type': 'categorical', 'padding_idx': 0, 'vocab_size': 173165}}
  {'item_id': {'source': '', 'type': 'categorical', 'padding_idx': 0, 'vocab_size': 63003}}
  {'cate_id': {'source': '', 'type': 'categorical', 'padding_idx': 0, 'vocab_size': 803}}
  {'item_history': {'source': '', 'type': 'sequence', 'feature_encoder': 'layers.MaskedAveragePooling()', 'share_embedding': 'item_id', 'padding_idx': 0, 'max_len': 50, 'vocab_size': 63003}}
  {'cate_history': {'source': '', 'type': 'sequence', 'feature_encoder': 'layers.MaskedAveragePooling()', 'share_embedding': 'cate_id', 'padding_idx': 0, 'max_len': 50, 'vocab_size': 803}}

parquet files:
   data/AmazonElectronics_x1/test.parquet
   data/AmazonElectronics_x1/train.parquet
   data/AmazonElectronics_x1/valid.parquet

sample head:


,label,cate_history,item_history,cate_id,item_id,user_id
0,1.0,"[125, 463, 50, 19, 32, 175, 90, 525, 608, 21, ...","[11398, 10290, 6360, 586, 28091, 4344, 5198, 4...",398,46511,10817
1,0.0,"[125, 463, 50, 19, 32, 175, 90, 525, 608, 21, ...","[11398, 10290, 6360, 586, 28091, 4344, 5198, 4...",171,47644,10817
2,1.0,"[107, 9, 5, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[32488, 1621, 1096, 1727, 0, 0, 0, 0, 0, 0, 0,...",1,2284,166552
3,0.0,"[107, 9, 5, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[32488, 1621, 1096, 1727, 0, 0, 0, 0, 0, 0, 0,...",232,37197,166552
4,1.0,"[59, 8, 25, 344, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[45006, 12406, 35334, 143, 0, 0, 0, 0, 0, 0, 0...",162,3680,169748


shape: (384806, 6)
item_history: len p50=50, p90=50, max=50
cate_history: len p50=50, p90=50, max=50
